In [ ]:
import pandas as pd

TARGETS = ["Fe", "Al", "As", "Pb", "Zn", "Hg", "Co", "V", "Ba", "Mn"]
metrics = ["r2_test", "mse_test", "r2_train", "mse_train"]
P = [1, 2, 3, 4]

df_vs = pd.read_excel("Results-vs.xlsx", sheet_name=0)
df_no_vs = pd.read_excel("Results-no-vs.xlsx", sheet_name=0)

with pd.ExcelWriter("analise.xlsx", engine="openpyxl") as writer:
    for target in TARGETS:
        vs = df_vs[df_vs["target"] == target]
        no_vs = df_no_vs[df_no_vs["target"] == target]

        # Estatísticas
        stats_vs = vs[metrics].agg(["mean", "std"])
        stats_no_vs = no_vs[metrics].agg(["mean", "std"])

        # Top 10 por r2_test
        top_vs = vs.sort_values("r2_test", ascending=False).head(10)
        top_no_vs = no_vs.sort_values("r2_test", ascending=False).head(10)

        # Montar uma única sheet por target, em blocos
        sheet_name = target

        row = 0
        pd.DataFrame({"VS": ["Média", "Desvio Padrão"]}).to_excel(
            writer, sheet_name=sheet_name, startrow=row, index=False
        )

        stats_vs.to_excel(writer, sheet_name=sheet_name, startrow=row+1)
        row += len(stats_vs) + 4

        pd.DataFrame({"NO-VS": ["Média", "Desvio Padrão"]}).to_excel(
            writer, sheet_name=sheet_name, startrow=row, index=False
        )

        stats_no_vs.to_excel(writer, sheet_name=sheet_name, startrow=row+1)
        row += len(stats_no_vs) + 4

        pd.DataFrame({"VS - Top 10 r2_test": []}).to_excel(
            writer, sheet_name=sheet_name, startrow=row, index=False
        )
        top_vs.to_excel(writer, sheet_name=sheet_name, startrow=row+1, index=False)
        row += len(top_vs) + 4

        pd.DataFrame({"NO-VS - Top 10 r2_test": []}).to_excel(
            writer, sheet_name=sheet_name, startrow=row, index=False
        )
        top_no_vs.to_excel(writer, sheet_name=sheet_name, startrow=row+1, index=False)


In [2]:
df_vs = pd.read_excel("Results-vs.xlsx", sheet_name=0)
df_no_vs = pd.read_excel("Results-no-vs.xlsx", sheet_name=0)

with pd.ExcelWriter("analise_2.xlsx", engine="openpyxl") as writer:
    for target in TARGETS:
        for p in sorted(df_vs["P"].dropna().unique()):
            vs = df_vs[(df_vs["target"] == target) & (df_vs["P"] == p)]
            no_vs = df_no_vs[(df_no_vs["target"] == target) & (df_no_vs["P"] == p)]

            if vs.empty and no_vs.empty:
                continue  # não cria aba vazia

            # Estatísticas
            stats_vs = vs[metrics].agg(["mean", "std"]) if not vs.empty else pd.DataFrame()
            stats_no_vs = no_vs[metrics].agg(["mean", "std"]) if not no_vs.empty else pd.DataFrame()

            # Top 10 por r2_test
            top_vs = vs.sort_values("r2_test", ascending=False).head(10) if not vs.empty else pd.DataFrame()
            top_no_vs = no_vs.sort_values("r2_test", ascending=False).head(10) if not no_vs.empty else pd.DataFrame()

            sheet_name = f"{target}_{int(p)}"
            row = 0

            # VS
            pd.DataFrame({"VS": ["Média", "Desvio Padrão"]}).to_excel(
                writer, sheet_name=sheet_name, startrow=row, index=False
            )
            if not stats_vs.empty:
                stats_vs.to_excel(writer, sheet_name=sheet_name, startrow=row+1)
                row += len(stats_vs) + 4
            else:
                row += 4

            # NO-VS
            pd.DataFrame({"NO-VS": ["Média", "Desvio Padrão"]}).to_excel(
                writer, sheet_name=sheet_name, startrow=row, index=False
            )
            if not stats_no_vs.empty:
                stats_no_vs.to_excel(writer, sheet_name=sheet_name, startrow=row+1)
                row += len(stats_no_vs) + 4
            else:
                row += 4

            # Top 10 VS
            pd.DataFrame({"VS - Top 10 r2_test": []}).to_excel(
                writer, sheet_name=sheet_name, startrow=row, index=False
            )
            if not top_vs.empty:
                top_vs.to_excel(writer, sheet_name=sheet_name, startrow=row+1, index=False)
                row += len(top_vs) + 4
            else:
                row += 4

            # Top 10 NO-VS
            pd.DataFrame({"NO-VS - Top 10 r2_test": []}).to_excel(
                writer, sheet_name=sheet_name, startrow=row, index=False
            )
            if not top_no_vs.empty:
                top_no_vs.to_excel(writer, sheet_name=sheet_name, startrow=row+1, index=False)


In [7]:
import pandas as pd

N = 5  # número de melhores modelos a considerar

df_vs = pd.read_excel("Results-vs.xlsx", sheet_name=0)
df_no_vs = pd.read_excel("Results-no-vs.xlsx", sheet_name=0)

# Unir as duas fontes (opcionalmente marque a origem)
df_vs["origem"] = "VS"
df_no_vs["origem"] = "NO-VS"
df_all = pd.concat([df_vs, df_no_vs], ignore_index=True)

sintese = []

for (target, p), g in df_all.groupby(["target", "P"]):
    top = g.sort_values("r2_test", ascending=False).head(N)

    # Avaliação dos critérios
    ok = (top["r2_test"] > 0.5) & (top["mse_test"] < 1.0)

    qtd_ok = ok.sum()
    status = "Satisfatório" if qtd_ok > 0 else "Não satisfatório"

    sintese.append({
        "target": target,
        "P": int(p),
        "avaliados": len(top),
        "bons_modelos": int(qtd_ok),
        "status": status,
        "melhor_r2": top["r2_test"].max(),
        "melhor_mse": top.loc[top["r2_test"].idxmax(), "mse_test"],
    })

df_sintese = pd.DataFrame(sintese).sort_values(["target", "P"])

print(df_sintese)


   target  P  avaliados  bons_modelos            status  melhor_r2  melhor_mse
0      Al  1          5             0  Não satisfatório    -1.6106   1076.1387
1      Al  2          5             0  Não satisfatório     0.5955   1439.8953
2      Al  3          5             0  Não satisfatório    -0.4274    824.2002
3      Al  4          5             0  Não satisfatório     0.6461    694.8133
4      As  1          5             5      Satisfatório     0.9102      0.0000
5      As  2          5             0  Não satisfatório     0.4599      0.0004
6      As  3          5             5      Satisfatório     0.7929      0.0005
7      As  4          5             5      Satisfatório     0.8873      0.0002
8      Ba  1          5             0  Não satisfatório     0.5493     25.4327
9      Ba  2          5             0  Não satisfatório     0.3528     33.0989
10     Ba  3          5             0  Não satisfatório     0.5239     24.8372
11     Ba  4          5             0  Não satisfató

In [8]:
df_sintese.to_excel("sintese_modelos.xlsx", index=False)